# Sato Weight 對神經纖維重建效果的影響（使用改進前的評估方法）

這個 notebook 使用**改進前**的評估方法測試 Sato 濾波器權重的影響。

## 評估方法對比

### 改進前（本 notebook）
- 使用**最大 Hausdorff 距離** (`directed_hausdorff`)
- **只使用節點座標**，不包含邊路徑點
- 對離群點敏感

### 改進後（另一個 notebook）
- 使用**平均 Hausdorff 距離**
- **包含節點 + 邊路徑點**
- 對離群點更穩健

## 背景

Sato 濾波器是一種管狀結構增強濾波器，可以增強圖像中的線狀結構（如神經纖維）。在預處理階段，我們可以選擇是否使用 Sato 濾波器以及其權重：

```python
fg_img = (1 - sato_weight) * rolling_fg + sato_weight * sato_normalized * 255
```

- `sato_weight = 0.0`: 完全不使用 Sato 濾波器（只用 rolling ball 背景減除）
- `sato_weight = 1.0`: 完全使用 Sato 濾波器結果
- `sato_weight = 0.5`: 混合使用

## 1. 環境設置

In [ ]:
import sys
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import directed_hausdorff

# 添加專案根目錄到 Python 路徑
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# 導入必要的模組
from tools.hierarchical_fragment_linking import HierarchicalFragmentLinker
from tools.compare_topologies import TopologyLoader
from tools.extract_dataset_topologies import TopologyExtractor

print("✓ 環境設置完成")

## 2. 實現改進前的比較器

使用最大 Hausdorff 距離和只使用節點的舊版本方法

In [ ]:
class OldMethodComparator:
    """
    改進前的拓撲比對器
    
    - 使用最大 Hausdorff 距離（directed_hausdorff）
    - 只使用節點座標，不包含邊路徑點
    """
    
    def __init__(self):
        pass
    
    def _extract_nodes_only(self, graph):
        """
        只提取節點座標（改進前的方法）
        
        Args:
            graph: NetworkX 圖
            
        Returns:
            節點座標數組，形狀 (N, 2)
        """
        if graph.number_of_nodes() == 0:
            return np.array([]).reshape(0, 2)
        
        nodes = list(graph.nodes())
        return np.array(nodes, dtype=float)
    
    def compute_max_hausdorff(self, graph1, graph2):
        """
        計算最大 Hausdorff 距離（改進前的方法）
        
        Args:
            graph1: 預測圖
            graph2: GT 圖
            
        Returns:
            dict: 包含距離和統計信息
        """
        # 只提取節點
        points1 = self._extract_nodes_only(graph1)
        points2 = self._extract_nodes_only(graph2)
        
        if len(points1) == 0 or len(points2) == 0:
            return {
                'status': 'failed',
                'error': '點集為空',
                'hausdorff_distance': None,
                'num_nodes1': graph1.number_of_nodes(),
                'num_nodes2': graph2.number_of_nodes(),
                'num_edges1': graph1.number_of_edges(),
                'num_edges2': graph2.number_of_edges(),
                'num_points1': len(points1),
                'num_points2': len(points2)
            }
        
        # 計算最大 Hausdorff 距離（雙向）
        d1 = directed_hausdorff(points1, points2)[0]
        d2 = directed_hausdorff(points2, points1)[0]
        max_hausdorff_dist = max(d1, d2)
        
        return {
            'status': 'success',
            'hausdorff_distance': float(max_hausdorff_dist),
            'hausdorff_1to2': float(d1),
            'hausdorff_2to1': float(d2),
            'num_nodes1': graph1.number_of_nodes(),
            'num_nodes2': graph2.number_of_nodes(),
            'num_edges1': graph1.number_of_edges(),
            'num_edges2': graph2.number_of_edges(),
            'num_points1': len(points1),  # 只有節點
            'num_points2': len(points2)   # 只有節點
        }
    
    def compare(self, graph1, graph2, label1="pred", label2="gt"):
        """
        比對兩個拓撲圖（使用改進前的方法）
        
        Args:
            graph1: 第一個圖
            graph2: 第二個圖
            label1: 第一個圖的標籤
            label2: 第二個圖的標籤
            
        Returns:
            dict: 比對結果
        """
        result = self.compute_max_hausdorff(graph1, graph2)
        result['label1'] = label1
        result['label2'] = label2
        return result

print("✓ 舊版本比對器實現完成")
print("  - 使用最大 Hausdorff 距離（directed_hausdorff）")
print("  - 只使用節點座標")

## 3. 載入測試數據

使用樣本 S1585-2_a 進行測試

In [ ]:
# 數據路徑
sample_id = "S1585-2_a"
data_dir = project_root / "data" / sample_id

image_path = data_dir / "image.png"
mask_path = data_dir / "mask.png"
annotation_path = data_dir / "annotation.png"
label_path = data_dir / "label.png"

# 驗證文件存在
for path in [image_path, mask_path, annotation_path, label_path]:
    if not path.exists():
        raise FileNotFoundError(f"文件不存在: {path}")

# 載入圖像
image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
annotation = cv2.imread(str(annotation_path), cv2.IMREAD_GRAYSCALE)
label = cv2.imread(str(label_path), cv2.IMREAD_GRAYSCALE)

print(f"✓ 載入完成")
print(f"  - 圖像尺寸: {image.shape}")
print(f"  - 遮罩尺寸: {mask.shape}")
print(f"  - 標註尺寸: {annotation.shape}")
print(f"  - GT 標籤尺寸: {label.shape}")

## 4. 提取 Ground Truth 拓撲

從 label.png 提取完整的 GT 拓撲，作為評測基準

In [ ]:
# 提取 GT 拓撲
extractor = TopologyExtractor()
gt_topology = extractor.extract_from_gt(label)

if gt_topology is None or gt_topology.number_of_nodes() == 0:
    raise ValueError("GT 拓撲提取失敗或為空")

print("✓ GT 拓撲提取完成")
print(f"  - 節點數: {gt_topology.number_of_nodes()}")
print(f"  - 邊數: {gt_topology.number_of_edges()}")
print(f"\n注意：改進前的方法只使用節點座標進行比對")

## 5. 測試不同的 Sato Weight

運行算法並記錄結果（使用改進前的評估方法）

In [ ]:
# 測試的 sato_weight 值
sato_weights = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

# 固定其他參數
common_params = {
    # 預處理參數
    'offset_px': 100,
    'rolling_ball_radius': 2,
    'opening_kernel_size': 3,
    # 重建參數
    'segment_length': 3.0,
    'search_radius_pathfinding': 50.0,
    'search_radius_endpoint_extension': 10.0,
    'max_angle_endpoint_extension': 75.0,
    'search_radius_mst': 20.0,
    'max_angle_mst': 90.0,
    'max_cost_threshold_mst': 0.75,
    'endpoint_extension_weight_discount': 0.5,
    'verbose': False
}

# 創建舊版本比對器
old_comparator = OldMethodComparator()

# 儲存結果
results = []

# 運行算法
print("\n開始測試不同的 sato_weight 值（使用改進前的評估方法）...\n")
print("評估方法：")
print("  - 最大 Hausdorff 距離（directed_hausdorff）")
print("  - 只使用節點座標\n")

for sato_weight in tqdm(sato_weights, desc="測試進度"):
    print(f"\n{'='*60}")
    print(f"測試 sato_weight = {sato_weight}")
    print(f"{'='*60}")
    
    try:
        # 創建連接器
        linker = HierarchicalFragmentLinker(
            sato_weight=sato_weight,
            **common_params
        )
        
        # 運行算法
        pred_topology = linker.run(image, mask, annotation)
        
        # 使用舊版本方法比對結果
        comparison = old_comparator.compare(
            pred_topology, 
            gt_topology,
            label1=f"sato_{sato_weight}",
            label2="GT"
        )
        
        # 記錄結果
        result = {
            'sato_weight': sato_weight,
            'num_nodes': comparison['num_nodes1'],
            'num_edges': comparison['num_edges1'],
            'num_points': comparison['num_points1'],  # 只有節點
            'hausdorff_distance': comparison['hausdorff_distance'],
            'hausdorff_1to2': comparison.get('hausdorff_1to2'),
            'hausdorff_2to1': comparison.get('hausdorff_2to1'),
            'status': comparison['status'],
            'topology': pred_topology
        }
        results.append(result)
        
        print(f"\n✓ 完成")
        print(f"  - 節點數: {result['num_nodes']} (只計算節點)")
        print(f"  - 邊數: {result['num_edges']}")
        print(f"  - 最大 Hausdorff 距離: {result['hausdorff_distance']:.4f} 像素")
        print(f"    - Pred→GT: {result['hausdorff_1to2']:.4f}")
        print(f"    - GT→Pred: {result['hausdorff_2to1']:.4f}")
        
    except Exception as e:
        print(f"\n✗ 失敗: {e}")
        import traceback
        traceback.print_exc()
        results.append({
            'sato_weight': sato_weight,
            'num_nodes': None,
            'num_edges': None,
            'num_points': None,
            'hausdorff_distance': None,
            'hausdorff_1to2': None,
            'hausdorff_2to1': None,
            'status': 'failed',
            'topology': None
        })

print("\n" + "="*60)
print("所有測試完成！")
print("="*60)

## 6. 結果分析

### 6.1 數據表格

In [ ]:
# 創建結果 DataFrame
df = pd.DataFrame([{
    'Sato Weight': r['sato_weight'],
    '節點數': r['num_nodes'],
    '邊數': r['num_edges'],
    '最大 Hausdorff': r['hausdorff_distance'],
    'Pred→GT': r.get('hausdorff_1to2'),
    'GT→Pred': r.get('hausdorff_2to1'),
    '狀態': r['status']
} for r in results])

# 添加 GT 參考
gt_row = pd.DataFrame([{
    'Sato Weight': 'GT',
    '節點數': gt_topology.number_of_nodes(),
    '邊數': gt_topology.number_of_edges(),
    '最大 Hausdorff': 0.0,
    'Pred→GT': 0.0,
    'GT→Pred': 0.0,
    '狀態': 'reference'
}])

df_with_gt = pd.concat([df, gt_row], ignore_index=True)

print("\n結果表格（改進前的方法）：\n")
print("評估方法：最大 Hausdorff 距離 + 只使用節點\n")
print(df_with_gt.to_string(index=False))

# 顯示完整表格（格式化）
display(df_with_gt.style.format({
    '最大 Hausdorff': '{:.4f}',
    'Pred→GT': '{:.4f}',
    'GT→Pred': '{:.4f}',
    '節點數': '{:.0f}',
    '邊數': '{:.0f}'
}).background_gradient(subset=['最大 Hausdorff'], cmap='RdYlGn_r'))

### 6.2 可視化比較

In [ ]:
# 過濾成功的結果
successful_results = [r for r in results if r['status'] == 'success']

if len(successful_results) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. 最大 Hausdorff 距離 vs Sato Weight
    ax1 = axes[0, 0]
    sato_vals = [r['sato_weight'] for r in successful_results]
    hausdorff_vals = [r['hausdorff_distance'] for r in successful_results]
    ax1.plot(sato_vals, hausdorff_vals, marker='o', linewidth=2, markersize=8, color='red')
    ax1.set_xlabel('Sato Weight', fontsize=12)
    ax1.set_ylabel('最大 Hausdorff 距離 (像素)', fontsize=12)
    ax1.set_title('最大 Hausdorff 距離 vs Sato Weight\n（改進前：只用節點）', 
                  fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # 找出最佳值
    best_idx = np.argmin(hausdorff_vals)
    ax1.plot(sato_vals[best_idx], hausdorff_vals[best_idx], 
             'g*', markersize=20, label=f'最佳: {sato_vals[best_idx]}')
    ax1.legend(fontsize=10)
    
    # 2. 雙向 Hausdorff 距離
    ax2 = axes[0, 1]
    h_1to2 = [r['hausdorff_1to2'] for r in successful_results]
    h_2to1 = [r['hausdorff_2to1'] for r in successful_results]
    ax2.plot(sato_vals, h_1to2, marker='s', linewidth=2, markersize=8, 
             color='blue', label='Pred→GT')
    ax2.plot(sato_vals, h_2to1, marker='^', linewidth=2, markersize=8,
             color='orange', label='GT→Pred')
    ax2.set_xlabel('Sato Weight', fontsize=12)
    ax2.set_ylabel('Hausdorff 距離 (像素)', fontsize=12)
    ax2.set_title('雙向 Hausdorff 距離', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=10)
    
    # 3. 節點數 vs Sato Weight
    ax3 = axes[1, 0]
    nodes_vals = [r['num_nodes'] for r in successful_results]
    ax3.plot(sato_vals, nodes_vals, marker='s', linewidth=2, markersize=8, color='green')
    ax3.axhline(y=gt_topology.number_of_nodes(), color='red', linestyle='--', 
                label=f'GT: {gt_topology.number_of_nodes()}')
    ax3.set_xlabel('Sato Weight', fontsize=12)
    ax3.set_ylabel('節點數', fontsize=12)
    ax3.set_title('節點數 vs Sato Weight', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.legend(fontsize=10)
    
    # 4. 邊數 vs Sato Weight
    ax4 = axes[1, 1]
    edges_vals = [r['num_edges'] for r in successful_results]
    ax4.plot(sato_vals, edges_vals, marker='^', linewidth=2, markersize=8, color='orange')
    ax4.axhline(y=gt_topology.number_of_edges(), color='red', linestyle='--',
                label=f'GT: {gt_topology.number_of_edges()}')
    ax4.set_xlabel('Sato Weight', fontsize=12)
    ax4.set_ylabel('邊數', fontsize=12)
    ax4.set_title('邊數 vs Sato Weight', fontsize=14, fontweight='bold')
    ax4.grid(True, alpha=0.3)
    ax4.legend(fontsize=10)
    
    plt.suptitle('改進前的評估方法：最大 Hausdorff 距離 + 只用節點', 
                 fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()
else:
    print("沒有成功的結果可供可視化")

### 6.3 統計分析

In [ ]:
if len(successful_results) > 0:
    print("\n" + "="*60)
    print("統計分析（改進前的方法）")
    print("="*60)
    print("\n評估方法：最大 Hausdorff 距離 + 只使用節點座標\n")
    
    # 最佳 sato_weight
    best_result = min(successful_results, key=lambda r: r['hausdorff_distance'])
    worst_result = max(successful_results, key=lambda r: r['hausdorff_distance'])
    
    print(f"最佳配置：")
    print(f"  - Sato Weight: {best_result['sato_weight']}")
    print(f"  - 最大 Hausdorff 距離: {best_result['hausdorff_distance']:.4f} 像素")
    print(f"    - Pred→GT: {best_result['hausdorff_1to2']:.4f}")
    print(f"    - GT→Pred: {best_result['hausdorff_2to1']:.4f}")
    print(f"  - 節點數: {best_result['num_nodes']} (GT: {gt_topology.number_of_nodes()})")
    print(f"  - 邊數: {best_result['num_edges']} (GT: {gt_topology.number_of_edges()})")
    
    print(f"\n最差配置：")
    print(f"  - Sato Weight: {worst_result['sato_weight']}")
    print(f"  - 最大 Hausdorff 距離: {worst_result['hausdorff_distance']:.4f} 像素")
    
    # 改進百分比
    improvement = (worst_result['hausdorff_distance'] - best_result['hausdorff_distance']) / worst_result['hausdorff_distance'] * 100
    print(f"\n改進幅度: {improvement:.2f}%")
    
    # Hausdorff 距離統計
    hausdorff_values = [r['hausdorff_distance'] for r in successful_results]
    print(f"\n最大 Hausdorff 距離統計：")
    print(f"  - 平均: {np.mean(hausdorff_values):.4f}")
    print(f"  - 中位數: {np.median(hausdorff_values):.4f}")
    print(f"  - 標準差: {np.std(hausdorff_values):.4f}")
    print(f"  - 範圍: [{np.min(hausdorff_values):.4f}, {np.max(hausdorff_values):.4f}]")
    
    print("\n" + "="*60)
    print("\n⚠️  注意：")
    print("  - 改進前的方法使用最大 Hausdorff 距離，對離群點敏感")
    print("  - 只使用節點座標，忽略了邊上的路徑點信息")
    print("  - 與改進後的平均 Hausdorff 距離結果不可直接比較")
    print("="*60)

### 6.4 保存結果

In [ ]:
if len(successful_results) > 0:
    # 創建輸出目錄
    output_dir = project_root / "output" / "sato_comparison_old_method" / sample_id
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # 保存所有拓撲
    loader = TopologyLoader()
    
    for result in results:
        if result['topology'] is not None:
            sato_weight = result['sato_weight']
            output_path = output_dir / f"sato_{sato_weight:.1f}.pkl"
            loader.save(result['topology'], output_path, format='pickle')
    
    # 保存 GT 拓撲
    gt_path = output_dir / "gt.pkl"
    loader.save(gt_topology, gt_path, format='pickle')
    
    # 保存結果表格
    csv_path = output_dir / "comparison_results_old_method.csv"
    df.to_csv(csv_path, index=False)
    
    print(f"\n✓ 結果已保存到: {output_dir}")
    print(f"  - 拓撲文件: {len([r for r in results if r['topology'] is not None])} 個")
    print(f"  - GT 拓撲: gt.pkl")
    print(f"  - 結果表格: comparison_results_old_method.csv")
    print(f"\n注意：這些結果使用改進前的評估方法（最大 Hausdorff + 只用節點）")

## 7. 與改進後方法的對比說明

### 改進前（本 notebook）

**評估方法：**
```python
# 1. 只提取節點
points = list(graph.nodes())

# 2. 計算最大 Hausdorff 距離
d1 = directed_hausdorff(points_pred, points_gt)[0]
d2 = directed_hausdorff(points_gt, points_pred)[0]
max_hausdorff = max(d1, d2)
```

**優點：**
- 計算簡單快速
- 容易理解和實現

**缺點：**
- 對離群點敏感（單個錯誤節點會導致很大的距離）
- 忽略邊路徑點，損失纖維形狀信息
- 可能無法準確反映整體重建質量

### 改進後（另一個 notebook）

**評估方法：**
```python
# 1. 提取節點 + 邊路徑點
points = list(graph.nodes())
for u, v, data in graph.edges(data=True):
    path = data.get('path') or data.get('path-coordinates')
    if path:
        points.extend(path)

# 2. 計算平均 Hausdorff 距離
d_avg_1to2 = mean(min_distance(p, points_gt) for p in points_pred)
d_avg_2to1 = mean(min_distance(p, points_pred) for p in points_gt)
avg_hausdorff = (d_avg_1to2 + d_avg_2to1) / 2
```

**優點：**
- 對離群點更穩健（使用平均而非最大值）
- 包含完整的纖維形狀信息（邊路徑點）
- 更準確反映整體重建質量

**缺點：**
- 計算開銷稍大
- 需要邊路徑點數據

### 數值差異預期

- **最大 Hausdorff** 通常比 **平均 Hausdorff** 大很多（可能數倍到數十倍）
- 兩個方法的**相對趨勢**應該類似（最佳 sato_weight 可能相同）
- 改進後的方法更適合評估重建質量

## 8. 結論

本 notebook 使用改進前的評估方法測試了 Sato 濾波器權重的影響。

雖然改進後的方法（平均 Hausdorff + 邊路徑點）更加準確和穩健，但了解改進前方法的結果有助於：

1. 驗證改進的有效性
2. 了解不同評估方法的差異
3. 與歷史結果進行比對

### 後續工作

- 對比兩種方法的數值差異
- 分析相對趨勢的一致性
- 在更多樣本上驗證結論